In [1]:
import wandb
api = wandb.Api()
import numpy as np

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [3]:
import json

In [4]:
runs = api.runs("ramithx1/Batch-Run-Maximum-Parsimony-A100-META-optimized")

In [5]:
def calc_avg_seq_similarity(leaves_data):
    length = leaves_data.shape[1]
    sim_percentages = []
    
    for i in range(0, leaves_data.shape[0]):
        for j in range(i+1, leaves_data.shape[0]):
            sim = (leaves_data[i] == leaves_data[j]).sum()/length
            
            sim_percentages.append(sim)
    
    sim_percentages = np.array(sim_percentages)
    
    return sim_percentages.mean()

In [6]:
def retrieve_exp(special_word, num_mutations):
    main_dict = {}
    errors = {}

    simulated = {}
    sankoffs = {}
    elves = {}

    for l in [4,8,16,32,64,128]:
        errors[str(l)] = []
        sankoffs[str(l)] = []
        elves[str(l)] = []
        simulated[str(l)] = []
        main_dict[str(l)] = []

        for run in runs:
            n_leaves = run.config['n_leaves']
            n_letters = run.config['n_letters']
            seq_length = run.config['seq_length']

            if(n_leaves == l and run.config['tLs']==[0, 0.005, 10, 50]):

                if("custom" in run.name or 'best cost' not in run.summaryMetrics.keys()):
                    continue

                if(special_word not in run.name or run.config['n_mutations'] != num_mutations):
                    continue

                assert n_letters == 20

                # print(run.config['epochs'])
                temp = run
                cost = run.summaryMetrics['best cost'] #run.summaryMetrics['traversal cost']
                gt   = run.summaryMetrics['sankoff_cost']
                sim = run.summaryMetrics['Groundtruth Total Loss']

                plotly_plot = run.summaryMetrics['Groundtruth Seq']
                
                
                if not os.path.exists(plotly_plot['path']):
                    run.file(plotly_plot['path']).download(replace=True) ## do just once

                with open(plotly_plot['path'], 'r') as file:
                    seq_data = np.array(json.load(file)['data'][0]['z'])

                    assert 2*n_leaves - 1 == seq_data.shape[0]
                    assert seq_length     == seq_data.shape[1]

                    leaves_data = seq_data[:n_leaves,:]

                val = calc_avg_seq_similarity(leaves_data)
                # print(f"{n_leaves} --> {val}")

                main_dict[str(l)].append([val, (cost - gt)/gt*100])

                errors[str(l)].append(cost - gt)
                sankoffs[str(l)].append(gt)
                elves[str(l)].append(cost)
                simulated[str(l)].append(sim)


                # print(run.summaryMetrics.keys())
                # print(run.summaryMetrics['sankoff_cost'])
                # print(run.summaryMetrics['traversal cost'])
                # print(run.history)
    
    for key in main_dict.keys():
        print(f"{key} --> {len(main_dict[key])}")
    
    return main_dict


### Perfect 50 Mutation Experiments

In [30]:
main_dict = retrieve_exp("Final Run", num_mutations = 50)

for key in [4,8,16,32,64,128]:
    
    sim    = np.array([x[0] for x in main_dict[str(key)]])
    errors = np.array([x[1] for x in main_dict[str(key)]])
    

    print(f"Leaves \t {key} \t Avg Similarity = \t {round(sim.mean(),3)}±{round(sim.std(),2)} \t Avg Error = \t {round(errors.mean(),3)}±{round(errors.std(),2)}")     

4 --> 10
8 --> 10
16 --> 10
32 --> 13
64 --> 10
128 --> 10
Leaves 	 4 	 Avg Similarity = 	 0.503±0.01 	 Avg Error = 	 0.0±0.0
Leaves 	 8 	 Avg Similarity = 	 0.379±0.01 	 Avg Error = 	 0.0±0.0
Leaves 	 16 	 Avg Similarity = 	 0.283±0.01 	 Avg Error = 	 0.007±0.02
Leaves 	 32 	 Avg Similarity = 	 0.209±0.01 	 Avg Error = 	 1.982±2.99
Leaves 	 64 	 Avg Similarity = 	 0.159±0.0 	 Avg Error = 	 4.373±0.46
Leaves 	 128 	 Avg Similarity = 	 0.123±0.0 	 Avg Error = 	 7.638±0.83


### Perfect 10 Mutation Experiments

In [7]:
main_dict = retrieve_exp("Perfect!M10", num_mutations = 10)

for key in [4,8,16,32,64]:
    
    sim    = np.array([x[0] for x in main_dict[str(key)]])
    errors = np.array([x[1] for x in main_dict[str(key)]])
    

    print(f"Leaves \t {key} \t Avg Similarity = \t {round(sim.mean(),3)}±{round(sim.std(),2)} \t Avg Error = \t {round(errors.mean(),3)}±{round(errors.std(),2)}")     

4 --> 10
8 --> 10
16 --> 2
32 --> 0
64 --> 3
128 --> 0
Leaves 	 4 	 Avg Similarity = 	 0.876±0.0 	 Avg Error = 	 0.0±0.0
Leaves 	 8 	 Avg Similarity = 	 0.825±0.0 	 Avg Error = 	 0.0±0.0
Leaves 	 16 	 Avg Similarity = 	 0.771±0.0 	 Avg Error = 	 0.0±0.0
Leaves 	 32 	 Avg Similarity = 	 nan±nan 	 Avg Error = 	 nan±nan
Leaves 	 64 	 Avg Similarity = 	 0.672±0.0 	 Avg Error = 	 13.944±0.47


/tmp/ipykernel_14214/1548626804.py:9: RuntimeWarning: Mean of empty slice.
  print(f"Leaves \t {key} \t Avg Similarity = \t {round(sim.mean(),3)}±{round(sim.std(),2)} \t Avg Error = \t {round(errors.mean(),3)}±{round(errors.std(),2)}")
/home/jupyter-ramith/.conda/envs/jax/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/jupyter-ramith/.conda/envs/jax/lib/python3.11/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/jupyter-ramith/.conda/envs/jax/lib/python3.11/site-packages/numpy/core/_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/jupyter-ramith/.conda/envs/jax/lib/python3.11/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype

In [8]:
main_dict

{'4': [[0.8763020833333334, 0.0],
  [0.876953125, 0.0],
  [0.880859375, 0.0],
  [0.8756510416666666, 0.0],
  [0.8756510416666666, 0.0],
  [0.8756510416666666, 0.0],
  [0.8736979166666666, 0.0],
  [0.8776041666666666, 0.0],
  [0.873046875, 0.0],
  [0.8704427083333334, 0.0]],
 '8': [[0.8271484375, 0.0],
  [0.8240792410714286, 0.0],
  [0.8327287946428571, 0.0],
  [0.8238002232142857, 0.0],
  [0.8238002232142857, 0.0],
  [0.82421875, 0.0],
  [0.8226841517857143, 0.0],
  [0.8274274553571429, 0.0],
  [0.8217075892857143, 0.0],
  [0.822265625, 0.0]],
 '16': [[0.7731770833333333, 0.0], [0.7688151041666667, 0.0]],
 '32': [],
 '64': [[0.6751689608134921, 14.081145584725538],
  [0.6720261346726191, 14.445331205107742],
  [0.6677497984871031, 13.306772908366534]],
 '128': []}